In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for compound screening analysis aggregation, join, scoring, and categorization
# Purpose: Aggregate, join, and categorize compound drug analysis data for screening insights
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script reads compound drug analysis data, filters for approved and valid records, aggregates metrics by therapeutic_area, joins the results, computes overall_score, categorizes potential, and displays comprehensive results.

# Required imports for PySpark DataFrame operations
from pyspark.sql import functions as F 
from pyspark.sql.types import DoubleType, LongType, StringType 

# -- Function: Aggregate compound drug analysis metrics by therapeutic_area
def aggregate_compound_metrics(df):
    """
    Aggregates compound drug analysis metrics by therapeutic_area.

    Args:
        df (DataFrame): Source DataFrame with compound drug analysis data.

    Returns:
        DataFrame: Aggregated metrics per therapeutic_area.
    """
    return df.groupBy("therapeutic_area").agg(
        F.avg("ic50").alias("avg_ic50"),
        F.avg("auc").alias("avg_auc"),
        F.avg("efficacy").alias("avg_efficacy"),
        F.sum("sample_size").alias("total_sample_size"),
        F.countDistinct("study_id").alias("study_count")
    )

# -- Function: Filter compound drug analysis for approved and valid records
def filter_approved_valid(df):
    """
    Filters compound drug analysis data for approved_flag == 1 and validation_status == 'valid'.

    Args:
        df (DataFrame): Source DataFrame.

    Returns:
        DataFrame: Filtered DataFrame.
    """
    return df.filter(
        (F.col("approved_flag") == 1) & (F.col("validation_status") == "valid")
    )

# -- Function: Compute overall_score and categorize potential
def add_score_and_category(df):
    """
    Adds overall_score and potential_category columns to DataFrame.

    Args:
        df (DataFrame): DataFrame with score1 to score5 columns.

    Returns:
        DataFrame: DataFrame with overall_score and potential_category.
    """
    df = df.withColumn(
        "overall_score",
        F.when(
            F.col("score1").isNull() | F.col("score2").isNull() | F.col("score3").isNull() | F.col("score4").isNull() | F.col("score5").isNull(),
            F.lit(None)
        ).otherwise(
            (F.col("score1") + F.col("score2") + F.col("score3") + F.col("score4") + F.col("score5")) / F.lit(5.0)
        )
    )
    df = df.withColumn(
        "potential_category",
        F.when(F.col("overall_score").isNull(), F.lit("Unknown"))
         .when((F.col("overall_score") >= 70) & (F.col("overall_score") <= 100), F.lit("High Potential"))
         .when((F.col("overall_score") >= 60) & (F.col("overall_score") < 70), F.lit("Moderate Potential"))
         .when(F.col("overall_score") < 60, F.lit("Low Potential"))
         .otherwise(F.lit("Unknown"))
    )
    return df

# -- Read compound_drug_analysis table with error handling for missing or invalid data
try:
    compound_df = spark.table("purgo_databricks.purgo_playground.compound_drug_analysis")
except Exception as e:
    # If table is missing or unreadable, create empty DataFrame with expected schema
    from pyspark.sql.types import StructType, StructField
    compound_schema = StructType([
        StructField("study_id", StringType(), True),
        StructField("compound_id", StringType(), True),
        StructField("mutation_id", StringType(), True),
        StructField("therapeutic_area", StringType(), True),
        StructField("drug_name", StringType(), True),
        StructField("ic50", DoubleType(), True),
        StructField("auc", DoubleType(), True),
        StructField("efficacy", DoubleType(), True),
        StructField("toxicity", DoubleType(), True),
        StructField("potency", DoubleType(), True),
        StructField("sample_size", LongType(), True),
        StructField("mutation_frequency", LongType(), True),
        StructField("mutation_severity", LongType(), True),
        StructField("compound_concentration", DoubleType(), True),
        StructField("cell_viability", DoubleType(), True),
        StructField("growth_inhibition", DoubleType(), True),
        StructField("result", StringType(), True),
        StructField("approved_flag", LongType(), True),
        StructField("validation_status", StringType(), True),
        StructField("status", StringType(), True),
        StructField("created_by", StringType(), True),
        StructField("score1", DoubleType(), True),
        StructField("score2", DoubleType(), True),
        StructField("score3", DoubleType(), True),
        StructField("score4", DoubleType(), True),
        StructField("score5", DoubleType(), True)
    ])
    compound_df = spark.createDataFrame([], compound_schema)

# -- Filter Analysis: Only rows with approved_flag == 1 and validation_status == 'valid'
filtered_df = filter_approved_valid(compound_df)

# -- Aggregation CTE: Group by therapeutic_area and calculate metrics
agg_df = aggregate_compound_metrics(filtered_df)

# -- Join Analysis: Join filtered data with aggregated metrics on therapeutic_area
joined_df = filtered_df.join(
    agg_df,
    on="therapeutic_area",
    how="left"
)

# -- Result Analysis: Compute overall_score and categorize results
final_df = add_score_and_category(joined_df)

# -- Select and order output columns to match target schema
output_columns = [
    "study_id", "compound_id", "mutation_id", "therapeutic_area", "drug_name", "ic50", "auc", "efficacy",
    "toxicity", "potency", "sample_size", "mutation_frequency", "mutation_severity", "compound_concentration",
    "cell_viability", "growth_inhibition", "result", "approved_flag", "validation_status", "status", "created_by",
    "score1", "score2", "score3", "score4", "score5", "avg_ic50", "avg_auc", "avg_efficacy", "total_sample_size", "study_count",
    "overall_score", "potential_category"
]

# -- Display the results with all columns and result analysis
final_df.select(*output_columns).show(truncate=False)

# -- End of script
